# Projet de Prallélisation maximale automatique de Systèmes d'exploitation

#### Ameylia CLEMENT 20230280   
#### Elisabeth SALAME 20230364   
#### Ninon CARISTAN 20232245  

Ce fichier contient toutes les classes et fonctions réalisées dans le cadre du projet.

In [ ]:
import graphviz
import random
import timeit

In [ ]:
class Task : 
    name = ""
    reads = []
    writes = []
    run = None

In [ ]:
class TaskSystem :
    def __init__(self, liste_t, dictionnaire_pre):
        self.liste_t = liste_t
        self.dictionnaire_pre = dictionnaire_pre
        self.dep_fin = {i : list(dictionnaire_pre[i]) for i in dictionnaire_pre}
        self.validateEntries()

    def parallelDict(self):
        ancetreTache = {t.name: list(self.getAncetresSequentiels(t.name)) for t in self.liste_t}
        for i in self.liste_t:
            for j in self.liste_t : 
                if i.name == j.name:
                    continue
                if (i.name in self.dep_fin[j.name]):
                    if any(var in i.writes for var in j.reads):
                        continue
                    if any(var in i.reads for var in j.writes):
                        continue
                    if any(var in i.writes for var in j.writes):
                        continue
                    ancetre_j = ancetreTache[j.name]
                    for ancetre in ancetre_j:
                        t_ancetre = next(t for t in self.liste_t if t.name == ancetre)
                        if any(var in i.writes for var in t_ancetre.reads):
                            continue
                        if any(var in i.reads for var in t_ancetre.writes):
                            continue
                        if any(var in i.writes for var in t_ancetre.writes):
                            continue
                    self.dep_fin[j.name].remove(i.name)

    def getAncetresSequentiels(self, nom_t):
        liste_v = set()
        liste_nv = list(self.dep_fin[nom_t])
        while liste_nv:
            tache = liste_nv.pop()
            if tache not in liste_v:
                liste_v.add(tache)
                liste_nv.extend(self.dep_fin[tache])
        return liste_v
    
    def getDependencies(self, nom_t):
        self.parallelDict()
        for i in self.liste_t:
            if i.name == nom_t:
                return self.dep_fin[i.name]
       
    def runSeq(self):
        liste_ex = [] 
        while len(liste_ex) < len(self.liste_t):
            for i in self.liste_t:
                if i.name in liste_ex : 
                    continue
                exe = True
                for dep in self.dep_fin[i.name]:
                        if dep not in liste_ex :
                            exe = False 
                            break
                if exe == True:
                    i.run()
                    liste_ex.append(i.name)
                    break
    
    def run(self):
        liste_ex = []
        liste_v = []
        self.parallelDict()
        while len(liste_ex)<len(self.liste_t):
            vague = []
            for i in self.liste_t:
                if i.name in liste_ex:
                    continue
                exe = True
                for dep in self.dep_fin[i.name]:
                    if dep not in liste_ex:
                        exe = False
                        break
                if exe == True:
                    vague.append(i)
            if(vague == []):
                print("Err : cycle détécté aucune tache executable")
                return
            for i in vague:
                print(i.name)
                i.run()
                liste_ex.append(i.name)
            liste_v.append(vague)
        return liste_v
    
    def validateEntries(self):
        checked = []
        for i in self.liste_t:
            if i.name in checked:
                print("Err : la tâche "+i.name+" n'est pas unique dans la liste des tâches")
                raise ValueError("Nom de tâche dupliqué")
            checked.append(i.name)

        tasks_names = {t.name for t in self.liste_t}
        for key, valueList in self.dep_fin.items():
            if key not in tasks_names: 
                print("Err : la tâche "+key+" du dictionnaire de précédences n'existe pas dans le système de tâches")
                raise ValueError("Tâche inexistante dans le dictionnaire de précédences")
            
            for i in valueList:
                if i not in tasks_names:
                    print("Err : la dépendance "+i+" référencée par "+key+" n'existe pas dans le système de tâches")
                    raise ValueError("Dépendance inexistante dans le dictionnaire de précédences")
                if key == i:
                    print("Err : la tâche "+key+" du dictionnaire de précédences ne peut pas dépendre d'elle-même")
                    raise ValueError("Auto dépendance")
            
        for i in self.liste_t:
            for j in self.liste_t : 
                if i.name == j.name:
                    continue
                ancetre_i = self.getAncetresSequentiels(i.name)
                ancetre_j = self.getAncetresSequentiels(j.name)
                if (i.name not in ancetre_j) and (j.name not in ancetre_i):
                    if any(var in i.writes for var in j.reads):
                        print("Err : la condition W1 intersection R2 = ensemble vide de Berstein n'est pas respectée pour les tâches "+i.name+" et "+j.name)
                        raise ValueError("Système indéterminé")
                    if any(var in i.reads for var in j.writes):
                        print("Err : la condition W2 intersection R1 = ensemble vide de Berstein n'est pas respectée pour les tâches "+i.name+" et "+j.name)
                        raise ValueError("Système indéterminé")
                    if any(var in i.writes for var in j.writes):
                        print("Err : la condition W1 intersection W2 = ensemble vide de Berstein n'est pas respectée pour les tâches "+i.name+" et "+j.name)
                        raise ValueError("Système indéterminé")
                
        liste_ex = [] 
        while len(liste_ex) < len(self.liste_t):
            found = False
            for i in self.liste_t:
                if i.name in liste_ex : 
                    continue
                exe = True
                for dep in self.dep_fin[i.name]:
                        if dep not in liste_ex :
                            exe = False 
                            break
                if exe == True:
                    liste_ex.append(i.name)
                    found = True 
                    break
            if found == False:
                print("Err : cycle détécté")
                raise ValueError("Cycle")
                    
        writesGlobal = {var for t in self.liste_t for var in t.writes}
        for i in self.liste_t:
            for j in i.reads: 
                if j not in writesGlobal: 
                    print("Err : la tâche est lue sans être initialisée")
                    raise ValueError("Lecture sans écriture")
                
    def draw(self):
        self.parallelDict()
        numG = input('Entrez le numéro de graphe : ')
        filename = "graphePrecedences"+numG
        graphe = graphviz.Digraph(comment = "Graphe des précédences")

        for i in self.liste_t:
            for j in self.dep_fin[i.name]:
                graphe.edge(j, i.name)

        graphe.render(filename, format = "png", cleanup = True)
        print(f"Graphe de précédence généré avec succès : {filename}.png")

    def detTestRnd(self, globals_dict, n_runs = 10, n_sets = 5): 
        vars_used = set()
        for t in self.liste_t : 
            vars_used.update(t.reads)
            vars_used.update(t.writes)

        print("Variables testées : ", vars_used)

        def restore_globals(values):
            for var, val in values.items() :
                globals_dict[var] = val

        for jeux in range(n_sets) : 
            print(f"\n == Jeu de valeurs {jeux+1} ==")
            initial_values = {v : random.randint(0, 1000) for v in vars_used}
            results = []

            for _ in range(n_runs) : 
                restore_globals(initial_values)
                random.shuffle(self.liste_t)
                print([t.name for t in self.liste_t])
                self.run()
                final_state = {v : globals_dict[v] for v in vars_used}
                results.append(final_state)
                print(f" Exécution {n_runs+1} -> {final_state}")

            reference = results[0]
            for r in results[1:] : 
                if r != reference : 
                    print(f"le système n'est pas déterministe : deux exécutions avec les mêmes valeurs initiales donnent deux résultats différent, ici, un résultat donne {reference} et {r}")
                    return False
	
        print(f"le système est déterministe : toutes les exécutions donnent le même résultat : {reference} et {r}")
        return True
    
    def parCost(self):
        seq_times = []
        par_times = []
        for i in range(5):
            self.runSeq()
            self.run()
        for i in range(20):
            start_time = timeit.default_timer()
            self.runSeq()
            seq_time = timeit.default_timer() - start_time
            seq_times.append(seq_time) 
            #Mesurer le temps d'execution parallele
            start_time = timeit.default_timer()
            self.run()
            par_time = timeit.default_timer() - start_time
            par_times.append(par_time)
        moyenne_seq = sum(seq_times) / len(seq_times)
        moyenne_par = sum(par_times) / len(par_times)
        print(f"Temps d'exécution séquentielle moyen en secondes : {moyenne_seq:.6f} secondes")
        print(f"Temps d'exécution parallèle moyen : {moyenne_par:.6f} secondes")